In [ ]:
import os
import gc
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer, RobertaModel, RobertaPreTrainedModel,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive

# ==========================================
# 1. CONFIGURATION & DIRECTORIES
# ==========================================
drive.mount('/content/drive')

# Model and Hyperparameters
MODEL_NAME = "microsoft/unixcoder-base"
MAX_LENGTH = 512
BATCH_SIZE = 128
LEARNING_RATE = 3e-5
EPOCHS = 1

# Paths - Update these to match your Drive folder structure
BASE_DIR = "/content/drive/MyDrive/SemEval_Models"
os.makedirs(BASE_DIR, exist_ok=True)

# Dataset labels
id2label = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
label2id = {v: k for k, v in id2label.items()}

# ==========================================
# 2. CUSTOM MODEL ARCHITECTURE
# ==========================================
class CustomUnixCoder(RobertaPreTrainedModel):
    """
    UnixCoder with a custom head concatenating Mean and Max Pooling.
    This architecture captures both global context and local features.
    """
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        seq_out = outputs[0] # [batch, seq_len, hidden_size]

        # Mean Pooling (ignoring padding)
        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

        # Max Pooling (ignoring padding by setting them to a very small number)
        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)

        # Concatenate and classify
        combined = torch.cat((mean_p, max_p), 1)
        logits = self.classifier(self.dropout(combined))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return (loss, logits) if loss is not None else logits

# ==========================================
# 3. DATA PREPARATION
# ==========================================
print("[INFO] Loading and tokenizing data...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# Load datasets (Ensure these files are in your BASE_DIR)
train_df = pd.read_parquet(os.path.join(BASE_DIR, 'train.parquet')).dropna(subset=['code', 'label'])
val_df = pd.read_parquet(os.path.join(BASE_DIR, 'validation.parquet')).dropna(subset=['code', 'label'])
test_df = pd.read_parquet(os.path.join(BASE_DIR, 'test.parquet'))

def tokenize_function(examples):
    return tokenizer(examples["code"], truncation=True, max_length=MAX_LENGTH)

# Prepare Datasets
ds_train = Dataset.from_pandas(train_df[['code', 'label']]).map(tokenize_function, batched=True, num_proc=4)
ds_val = Dataset.from_pandas(val_df[['code', 'label']]).map(tokenize_function, batched=True, num_proc=4)
ds_test = Dataset.from_pandas(test_df).map(tokenize_function, batched=True, num_proc=4)

# Formatting for Trainer
for ds in [ds_train, ds_val, ds_test]:
    if "label" in ds.column_names:
        ds = ds.rename_column("label", "labels")
    ds.set_format("torch")

# ==========================================
# 4. TRAINING EXECUTION
# ==========================================
model = CustomUnixCoder.from_pretrained(MODEL_NAME, num_labels=11)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='macro')
    return {'f1_macro': f1, 'accuracy': accuracy_score(p.label_ids, preds)}

training_args = TrainingArguments(
    output_dir=os.path.join(BASE_DIR, "unixcoder_p1"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True, # Enable mixed precision for speed
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

print("🚀 Starting Training...")
trainer.train()

# ==========================================
# 5. SUBMISSION GENERATION
# ==========================================
print("\n[INFO] Generating submission.csv...")
test_out = trainer.predict(ds_test)
test_preds = np.argmax(test_out.predictions, axis=1)

submission = pd.DataFrame({
    'ID': test_df.get('ID', test_df.index),
    'label': test_preds
})
submission.to_csv(os.path.join(BASE_DIR, 'submission.csv'), index=False)
print(f"✅ Submission saved to {os.path.join(BASE_DIR, 'submission.csv')}")

# ==========================================
# 6. HARD NEGATIVE MINING & ANALYSIS
# ==========================================
def mine_and_analyze(trainer, df, ds, id2label):
    print("\n" + "="*50)
    print("⛏️ HARD NEGATIVE MINING & ANALYSIS")
    print("="*50)

    # 1. Prediction on full train set to get losses
    out = trainer.predict(ds)
    logits = torch.tensor(out.predictions)
    labels = torch.tensor(out.label_ids)

    loss_fct = nn.CrossEntropyLoss(reduction='none')
    losses = loss_fct(logits, labels)

    # 2. Select top 20% by loss
    sorted_losses, sorted_indices = torch.sort(losses, descending=True)
    hard_count = int(len(ds) * 0.20)
    hard_indices = sorted_indices[:hard_count].tolist()

    # 3. Analyze Hard Negatives
    hard_df = df.iloc[hard_indices].copy()
    total = len(hard_df)
    human_count = len(hard_df[hard_df['label'] == 0])
    ai_count = total - human_count

    print(f"\n📊 SUMMARY:")
    print(f"Total Hard Examples: {total}")
    print(f"Human: {human_count} ({human_count/total:.2%})")
    print(f"AI: {ai_count} ({ai_count/total:.2%})")

    print("\n📈 CLASS-BASED DISTRIBUTION OF FAILURES:")
    label_counts = hard_df['label'].value_counts().sort_index()
    for lid, count in label_counts.items():
        name = id2label[lid]
        print(f" - {name.ljust(12)}: {count} examples ({count/total:.2%})")

    # 4. Save results
    results = {
        "hard_indices": hard_indices,
        "distribution": {id2label[k]: int(v) for k, v in label_counts.to_dict().items()}
    }
    with open(os.path.join(BASE_DIR, "hard_negatives.json"), "w") as f:
        json.dump(results, f, indent=4)
    print(f"\n✅ Analysis saved to {os.path.join(BASE_DIR, 'hard_negatives.json')}")

mine_and_analyze(trainer, train_df, ds_train, id2label)